In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
import sys

DRIVE = "/content/drive/MyDrive/volatility-forecast"
drive.mount("/content/drive", force_remount=True)
os.chdir(DRIVE)

Mounted at /content/drive


In [ ]:
raw = pd.read_csv('data/processed/dataset.csv')
print("raw shape:", raw.shape)
print("columns  :", raw.columns.tolist())

FEATURES = ['rv1', 'rv5', 'rv21']   # order = beta [d, w, m]; fix if names differ
TARGET   = 'y_rv21'                  # h21 forward target; fix if differs
DATECOL  = 'Date'                    # row-1008 sanity check only; fix if differs

# rebuild work frame: drop head/tail NaN -> 3831, 0-indexed (matches fold constants)
wf = raw.dropna(subset=FEATURES + [TARGET]).reset_index(drop=True)
print("work frame:", wf.shape, "(expect 3831)")
print("row 1008  :", wf.loc[1008, DATECOL], "(expect 2015-02-05)")

STATIC_END_OLD = 1008
STATIC_END_NEW = 987   # embargo: last train row 986, label 986+21=1007 < 1008 (first OOS)

def refit(end):
    sub = wf.iloc[:end]
    return np.asarray(fit_har(sub[FEATURES].to_numpy(float),
                              sub[TARGET].to_numpy(float)))

beta_old, beta_new = refit(STATIC_END_OLD), refit(STATIC_END_NEW)
locked = np.array([0.0048, 0.0258, 0.201, 0.2949])
lab = ['intercept', 'beta_d', 'beta_w', 'beta_m']

print("\n           " + "".join(f"{l:>11}" for l in lab))
print("locked    :" + "".join(f"{v:11.4f}" for v in locked))
print("refit 1008:" + "".join(f"{v:11.4f}" for v in beta_old))
print("refit  987:" + "".join(f"{v:11.4f}" for v in beta_new))
print("delta     :" + "".join(f"{v:11.5f}" for v in beta_new - beta_old))
print("rel %     :" + "".join(f"{v:11.2f}" for v in 100*(beta_new - beta_old)/beta_old))

assert wf.shape[0] == 3831, "work frame != 3831 -> trim/NaN handling differs"
assert np.allclose(beta_old, locked, atol=5e-4), \
    "refit 1008 != locked -> check FEATURES/TARGET order or NaN"
print("\nOK: work frame + fit path verified -> 987 delta trustworthy")

raw shape: (3873, 22)
columns  : ['Date', 'qqq_ret', 'hyg_ret', 'lqd_ret', 'tlt_ret', 'gld_ret', 'vix_lvl', 'vix_chg', 'tnx_lvl', 'tnx_chg', 'irx_lvl', 'irx_chg', 'slope_lvl', 'slope_chg', 'credit_lvl', 'credit_chg', 'rv1', 'rv5', 'rv21', 'y_rv1', 'y_rv5', 'y_rv21']
work frame: (3831, 22) (expect 3831)
row 1008  : 2015-02-05 (expect 2015-02-05)

             intercept     beta_d     beta_w     beta_m
locked    :     0.0048     0.0258     0.2010     0.2949
refit 1008:     0.0048     0.0258     0.2010     0.2949
refit  987:     0.0048     0.0261     0.2056     0.2937
delta     :    0.00000    0.00033    0.00457   -0.00125
rel %     :       0.06       1.26       2.27      -0.43

OK: work frame + fit path verified -> 987 delta trustworthy


In [ ]:
path = 'src/retrain.py'
src = open(path).read()
old = "df.iloc[:start]"
new = "df.iloc[:start - label_delay]"
assert src.count(old) == 1, f"expected 1 match, found {src.count(old)} -- STOP, inspect"
open(path, 'w').write(src.replace(old, new))

import sys
for m in list(sys.modules):
    if m.startswith('src'):
        del sys.modules[m]
print("patched line:")
print([l.strip() for l in open(path).read().splitlines()
       if 'fit_har_frame(df.iloc[:start' in l])

patched line:
['beta = fit_har_frame(df.iloc[:start - label_delay])   # initial model: static HAR on first `start` rows']


In [10]:
import inspect
from src.conformal import aci_stream_coverage
print(inspect.getsource(aci_stream_coverage))

def aci_stream_coverage(resid, sigma, test_blocks, cal_window=252,
                        alpha=0.1, gamma=0.01, a_lo=1e-3, a_hi=1-1e-3):
    """Adaptive Conformal Inference (Gibbs & Candès 2021) run as one
    continuous online stream, with coverage aggregated per fold afterward.

    Normalized score s_i = |resid_i| / sigma_i. At each step t (over the
    union of all test blocks, in time order) the band uses the
    (1 - alpha_t) quantile of the trailing cal_window normalized scores;
    alpha_t is then updated by
        alpha_{t+1} = alpha_t + gamma * (alpha - err_t),
        err_t = 1[Y_t not covered].
    alpha_t carries across fold boundaries (true online behavior); it is
    NOT reset per fold. Coverage is then averaged within each fold.

    Parameters
    ----------
    resid, sigma : np.ndarray, shape (T,)   signed residuals and local scale.
    test_blocks : dict {fold:int -> np.ndarray of absolute test indices}.
    cal_window : int    trailing calibration length for eac

In [11]:
path = 'src/conformal.py'
src = open(path).read()

reps = [
    # signature: add delay param
    ("test_blocks, cal_window=252,",
     "test_blocks, cal_window=252, delay=21,"),
    # docstring: recursion now uses matured err
    ("alpha_{t+1} = alpha_t + gamma * (alpha - err_t),",
     "alpha_{t+1} = alpha_t + gamma * (alpha - err_{t-delay}),"),
    # docstring: param line
    ("cal_window : int    trailing calibration length for each step's quantile.",
     "cal_window : int    trailing calibration length for each step's quantile.\n"
     "    delay : int         label maturity lag (h21 -> 21). At day t only indices\n"
     "                        s <= t-delay are visible; delay=0 reproduces the\n"
     "                        original (leaky) behavior for ablation."),
    # quantile window: matured scores only
    ("        cal = score[t - cal_window:t]            # trailing window, excludes t",
     "        m = t - delay                            # newest index matured by day t\n"
     "        cal = score[m - cal_window:m]            # matured trailing window"),
    # err feedback: delayed by `delay`
    ("        err = 0.0 if covered else 1.0\n"
     "        alpha_t = np.clip(alpha_t + gamma * (alpha - err), a_lo, a_hi)",
     "        # delayed feedback: only err_{t-delay} matures today; err_t is not\n"
     "        # observable until t+delay -- mirrors the stage-08 label_delay rule.\n"
     "        if m in covered_at:                      # first `delay` OOS days: no update\n"
     "            err = 0.0 if covered_at[m] else 1.0\n"
     "            alpha_t = np.clip(alpha_t + gamma * (alpha - err), a_lo, a_hi)"),
]

for old, new in reps:
    n = src.count(old)
    assert n == 1, f"match count {n} != 1 for: {old[:60]!r} -- STOP"
    src = src.replace(old, new)
open(path, 'w').write(src)

import sys
for m in list(sys.modules):
    if m.startswith('src'):
        del sys.modules[m]
from src.conformal import aci_stream_coverage
import inspect
sig = inspect.signature(aci_stream_coverage)
print("new signature:", sig)

new signature: (resid, sigma, test_blocks, cal_window=252, delay=21, alpha=0.1, gamma=0.01, a_lo=0.001, a_hi=0.999)
